# Submission Report (Phase 1)

This notebook checks that the team submission file is valid for Kaggle format.

It verifies:
- column names and order,
- row count,
- query ID ordering,
- JSON format of `relevant_doc_ids`,
- top-k size consistency.


In [ ]:
from pathlib import Path
import json
import shutil

import pandas as pd

BASE_DIR = Path('../..')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs'

TEMPLATE_PATH = DATA_DIR / 'submission.csv'
FINAL_PATH = OUTPUT_DIR / 'solutions_SeaFour.csv'

BM25_PATH = OUTPUT_DIR / 'solutions_SeaFour_bm25.csv'
TFIDF_PATH = OUTPUT_DIR / 'solutions_SeaFour_tfidf.csv'


In [ ]:
def validate_submission_file(submission_path: Path, template_path: Path) -> dict:
    template_df = pd.read_csv(template_path)
    submission_df = pd.read_csv(submission_path)

    result = {
        'submission': str(submission_path.name),
        'exists': submission_path.exists(),
        'column_match': list(submission_df.columns) == list(template_df.columns),
        'row_count_match': len(submission_df) == len(template_df),
        'query_id_order_match': (
            submission_df.iloc[:, 0].astype(str).tolist() ==
            template_df.iloc[:, 0].astype(str).tolist()
        ),
        'valid_json_lists': True,
        'min_k': None,
        'max_k': None,
    }

    lengths = []
    try:
        for raw in submission_df.iloc[:, 1].astype(str):
            parsed = json.loads(raw)
            if not isinstance(parsed, list):
                result['valid_json_lists'] = False
                break
            lengths.append(len(parsed))
    except Exception:
        result['valid_json_lists'] = False

    if lengths:
        result['min_k'] = min(lengths)
        result['max_k'] = max(lengths)

    result['is_valid'] = all([
        result['column_match'],
        result['row_count_match'],
        result['query_id_order_match'],
        result['valid_json_lists'],
    ])

    return result


In [ ]:
checks = []
for path in [BM25_PATH, TFIDF_PATH, FINAL_PATH]:
    if path.exists():
        checks.append(validate_submission_file(path, TEMPLATE_PATH))
    else:
        checks.append({'submission': path.name, 'exists': False})

pd.DataFrame(checks)


In [ ]:
# If final file is missing, create it from BM25 output.
# This matches the pipeline default FINAL_MODEL='bm25'.
if not FINAL_PATH.exists():
    shutil.copy2(BM25_PATH, FINAL_PATH)

print('Final file exists:', FINAL_PATH.exists())
print('Final path:', FINAL_PATH.resolve())


In [ ]:
final_check = validate_submission_file(FINAL_PATH, TEMPLATE_PATH)
pd.DataFrame([final_check])


In [ ]:
final_df = pd.read_csv(FINAL_PATH)
final_df.head()


## Result

If `is_valid == True`, then `outputs/solutions_SeaFour.csv` matches Kaggle template requirements
(`submission.csv`) for phase-1 style submission formatting.
